In [1]:
import numpy as np
import trimesh
import torch

In [ ]:
def scale_mesh_3d(mesh, edge_factors):
    """
    Scales the edges of the mesh in 3D space by given factors.

    :param mesh: The trimesh mesh instance.
    :param edge_factors: A list or array of scale factors for each edge.
    :return: Scaled mesh.
    """
    # Create a copy of the vertices to manipulate
    vertices = np.array(mesh.vertices)
    # Iterate over each edge and scale
    for i, edge in enumerate(mesh.edges_unique):
        # Scale factor for this edge
        scale_factor = edge_factors[i % len(edge_factors)]
        # Get the vertex positions for the edge
        v0, v1 = vertices[edge[0]], vertices[edge[1]]
        # Calculate the midpoint
        midpoint = (v0 + v1) / 2.0
        # Calculate the direction vector for the edge
        direction = v1 - v0
        # Scale the edge vertices in 3D
        vertices[edge[0]] = midpoint - direction * scale_factor / 2.0
        vertices[edge[1]] = midpoint + direction * scale_factor / 2.0
    # Update the mesh with the new vertices
    mesh.vertices = vertices
    return mesh

# Load the mesh
mesh = trimesh.load_mesh('path_to_your_mesh.ply')

# Edge scaling factors, with random variation between 1.05 and 1.15
random_factors = 1.05 + np.random.rand(len(mesh.edges_unique)) * 0.1

# Scale the mesh edges in 3D
scaled_mesh = scale_mesh_3d(mesh, random_factors)

# Save the modified mesh
scaled_mesh.export('scaled_mesh.ply')

In [5]:
# Load the mesh
mesh = trimesh.load_mesh('results/tl_out/orig_upper_garment.ply')

# Create a scaling matrix
scale_factor = 1.2
scaling_matrix = np.array([
    [scale_factor, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, scale_factor, 0],  # Z-axis is not scaled
    [0, 0, 0, 1]
])

# Apply the scaling
mesh.apply_transform(scaling_matrix)

# Save the modified mesh
mesh.export('modified_mesh.ply')

b'ply\nformat binary_little_endian 1.0\ncomment https://github.com/mikedh/trimesh\nelement vertex 2278\nproperty float x\nproperty float y\nproperty float z\nproperty uchar red\nproperty uchar green\nproperty uchar blue\nproperty uchar alpha\nelement face 4464\nproperty list uchar int vertex_indices\nend_header\n\xd1x\xce\xbd\x97\xb0\x87>\xb5\x91R<}\x00\x00\xff\x08\xab8\xbe\xffO\xab=\xed\xa4\xe9=}\x00\x00\xffa13\xbe\xe8\x9b\x90=\xb1\xb0\xff=}\x00\x00\xff+\xb5-\xbe\n.\xa7=%N\n>}\x00\x00\xffNN1\xbe0\x80\xc1=~\xc9\x00>}\x00\x00\xff\xbfJ\r\xbe&:r;\xa4\n\xe1=}\x00\x00\xff\xa4]\x10\xbe{Vi\xbc\x13\x1f\xe1=}\x00\x00\xff\x9b\xb4\xfc\xbd\'\x83c\xbc\x8dM\x04>}\x00\x00\xff\xe0\x1b\xf7\xbdW\x02\x82;\xafB\x03>}\x00\x00\xff\xdd+\x06\xbe\x9a`\x1a>\xf0\xbc\xfa=}\x00\x00\xff\x8c\x94\xea\xbd\x15\xdd\x1b>\x0ft\x03>}\x00\x00\xff\x94|\xed\xbd!1+>\xfec\xe9=}\x00\x00\xffH#\x08\xbe\xe4\xe3)>G\x8d\xde=}\x00\x00\xffWy\xae=T\xbf\x90>\x11\x1f\xac;}\x00\x00\xff\xd5\xcc6\xbe\xce\xce\x06>J\xc2\xbe=}\x00\x00\xff\x91\x

In [ ]:
# Load the mesh
mesh = trimesh.load('output/orig_upper_garment.ply')

# Get edges from the mesh
edges = mesh.edges

# Get unique vertices in the edges
unique_vertices = np.unique(edges)

# Create a mapping for unique vertex indices to their new positions after scaling
vertex_map = {}

# For each vertex, compute a random scaling factor and apply it
for vertex_index in unique_vertices:
    # Random scaling factors between 1.15 and 1.25 for the X and Y directions
    scale_factor_x = np.random.uniform(1.15, 1.25)
    scale_factor_y = np.random.uniform(1.15, 1.25)

    # Get the original vertex position
    original_vertex = mesh.vertices[vertex_index]

    # Apply the scaling to the vertex position
    # Note: We're not scaling the Z position
    scaled_vertex = original_vertex * [scale_factor_x, 1, scale_factor_y]

    # Store the scaled vertex position in the mapping
    vertex_map[vertex_index] = scaled_vertex

# Update the vertices in the mesh with the new scaled positions
for vertex_index, new_position in vertex_map.items():
    mesh.vertices[vertex_index] = new_position

mesh.export('modified_mesh.ply')

In [1]:
def calculate_triangle_areas(vertices, triangles):
    v0 = vertices[triangles[:, 0]]
    v1 = vertices[triangles[:, 1]]
    v2 = vertices[triangles[:, 2]]

    area = torch.linalg.norm(torch.cross(v1 - v0, v2 - v0), dim=1) / 2
    return area


def calculate_target_stretches(parameterized_vertices, triangles, initial_areas):
    parameterized_areas = calculate_triangle_areas(parameterized_vertices, triangles)
    target_stretches = parameterized_areas / initial_areas
    return target_stretches
    

def objective_function_parallel(vertices, triangles, initial_areas, target_stretches):
    v0 = vertices[triangles[:, 0]]
    v1 = vertices[triangles[:, 1]]
    v2 = vertices[triangles[:, 2]]

    areas = torch.linalg.norm(torch.cross(v1 - v0, v2 - v0), dim=1) / 2
    current_stretches = areas / initial_areas
    stretch_differences = current_stretches - target_stretches

    return torch.sum(stretch_differences ** 2)


def compute_cotangent_weights_parallel(vertices, triangles):
    num_vertices = vertices.size(0)
    adjacency_matrix = torch.zeros((num_vertices, num_vertices), dtype=torch.float32)
    degree_matrix = torch.zeros((num_vertices, num_vertices), dtype=torch.float32)

    v_i = vertices[triangles[:, 0]]  # vertices at first corner
    v_j = vertices[triangles[:, 1]]  # vertices at second corner
    v_k = vertices[triangles[:, 2]]  # vertices at third corner

    vki = v_i - v_k  # Vector from vertex k to i
    vkj = v_j - v_k  # Vector from vertex k to j

    # Compute cotangents of angles between vectors
    dot_product = (vki * vkj).sum(-1)  # Dot product of vectors
    cross_norm = torch.linalg.norm(torch.cross(vki, vkj), dim=1)  # Norm of cross product
    cot_ki_j = dot_product / cross_norm  # Cotangent of angle

    # Update adjacency matrix
    i, j, k = triangles[:, 0], triangles[:, 1], triangles[:, 2]
    adjacency_matrix[i, j] += cot_ki_j
    adjacency_matrix[j, i] += cot_ki_j

    # Update degree matrix
    degree_matrix[i, i] += cot_ki_j
    degree_matrix[j, j] += cot_ki_j

    return adjacency_matrix, degree_matrix


def compute_laplacian_loss(vertices, laplacian_matrix, original_laplacian_coords):
    current_laplacian_coords = torch.matmul(laplacian_matrix, vertices)
    laplacian_loss = torch.mean((current_laplacian_coords - original_laplacian_coords) ** 2)
    return laplacian_loss


def compute_laplacian_matrix(adjacency_matrix, degree_matrix):
    return degree_matrix - adjacency_matrix


def regularization_loss(vertices, original_vertices, gamma):
    return torch.sum((vertices - original_vertices) ** 2)



mesh = trimesh.load('results/optim_in/orig_front_shirt.obj')
vertices = torch.tensor(mesh.vertices, dtype=torch.float32, requires_grad=True) # type: ignore
triangles = torch.tensor(mesh.faces, dtype=torch.int32) # type: ignore

parameterized_mesh = trimesh.load('results/optim_in/param_front_shirt.obj')
parameterized_vertices = torch.tensor(parameterized_mesh.vertices, dtype=torch.float32) # type: ignore

orig_verts = vertices.clone().detach()

num_iterations = 500
alpha = 1.0
beta = 10.
gamma = 0.

initial_areas = calculate_triangle_areas(orig_verts, triangles)
adjacency_matrix, degree_matrix = compute_cotangent_weights_parallel(orig_verts, triangles)
laplacian_matrix = compute_laplacian_matrix(adjacency_matrix, degree_matrix)
original_laplacian_coords = torch.matmul(laplacian_matrix, orig_verts)
target_stretches = calculate_target_stretches(parameterized_vertices, triangles, initial_areas)

target_stretches = torch.ones_like(target_stretches) * 1.2

optimizer = torch.optim.Adam([vertices], lr=0.0025)

for iteration in range(num_iterations):
    optimizer.zero_grad()

    stretch_loss = objective_function_parallel(vertices, triangles, initial_areas, target_stretches)
    laplacian_loss = compute_laplacian_loss(vertices, laplacian_matrix, original_laplacian_coords)
    reg_loss = regularization_loss(vertices, orig_verts, gamma)
    
    total_loss = alpha * stretch_loss + beta * laplacian_loss + gamma * reg_loss

    total_loss.backward()
    optimizer.step()

    if iteration % 5 == 0:
        print(f"Iteration {iteration}, Loss: {total_loss.item():.5f} (Stretch: {stretch_loss:.5f}, Laplacian: {laplacian_loss:.5f}, Reg: {reg_loss:.5f})")
    
    if total_loss < 0.01:
        break

optimized_mesh = trimesh.Trimesh(vertices=vertices.detach().numpy(), faces=triangles.numpy())
optimized_mesh.export('results/optim_out/optimized_fron_shirt.ply')


NameError: name 'trimesh' is not defined

## Jacobian-based optimization

In [1]:
import numpy as np
import trimesh
import torch
from scipy.spatial import KDTree


def compute_jacobians(vertices, triangles, warp_direction, weft_direction):
    # Extract vertices of the triangles
    v0s = vertices[triangles[:, 0]]
    v1s = vertices[triangles[:, 1]]
    v2s = vertices[triangles[:, 2]]

    # Compute edge vectors for all triangles simultaneously
    edge1s = v1s - v0s
    edge2s = v2s - v0s

    # Project edge vectors onto warp and weft directions for all triangles
    J_warp = torch.stack((torch.sum(edge1s * warp_direction, dim=1),
                          torch.sum(edge2s * warp_direction, dim=1)), dim=1)
    J_weft = torch.stack((torch.sum(edge1s * weft_direction, dim=1),
                          torch.sum(edge2s * weft_direction, dim=1)), dim=1)

    # Form the Jacobian matrices from the projections
    J = torch.stack((J_warp, J_weft), dim=2)
    return J

def jacobian_loss_function(initial_jacobians, target_jacobians):
    # Compute the Frobenius norm of the difference between Jacobians for all triangles
    loss = torch.norm(initial_jacobians - target_jacobians, dim=(1, 2))
    return torch.mean(loss)

In [2]:


if __name__ == '__main__':
    # Load meshes, setup tensors, etc.
    mesh = trimesh.load('results/optim_in/orig_front_shirt_0.001.ply')
    vertices = torch.tensor(mesh.vertices, dtype=torch.float32, requires_grad=True)
    triangles = torch.tensor(mesh.faces, dtype=torch.int32)

    parameterized_mesh = trimesh.load('results/optim_in/param_front_shirt.obj')
    parameterized_vertices = torch.tensor(parameterized_mesh.vertices, dtype=torch.float32)

    # Define warp and weft directions (these should be normalized)
    warp_direction = torch.tensor([1, 0, 0], dtype=torch.float32)  # Example direction
    weft_direction = torch.tensor([0, 1, 0], dtype=torch.float32)

    # Compute Jacobians for all triangles in both meshes
    initial_jacobians = compute_jacobians(vertices, triangles, warp_direction, weft_direction)
    target_jacobians = compute_jacobians(parameterized_vertices, triangles, warp_direction, weft_direction)

    num_iterations = 100
    optimizer = torch.optim.Adam([vertices], lr=0.001)
    for iteration in range(num_iterations):
        optimizer.zero_grad()

        # Recompute current Jacobians based on updated vertices
        current_jacobians = compute_jacobians(vertices, triangles, warp_direction, weft_direction)
        jacobian_loss = jacobian_loss_function(current_jacobians, target_jacobians)
        
        jacobian_loss.backward()
        optimizer.step()

        if iteration % 5 == 0:
            print(f"Iteration {iteration}, Jacobian Loss: {jacobian_loss.item()}")

    # Save or further process the optimized mesh
    optimized_mesh = trimesh.Trimesh(vertices=vertices.detach().numpy(), faces=triangles.numpy())
    optimized_mesh.export('optimized_mesh.ply')

Iteration 0, Jacobian Loss: 0.015601989813148975
Iteration 5, Jacobian Loss: 0.013448622077703476
Iteration 10, Jacobian Loss: 0.012218751944601536
Iteration 15, Jacobian Loss: 0.011277349665760994
Iteration 20, Jacobian Loss: 0.010490589775145054
Iteration 25, Jacobian Loss: 0.009836980141699314
Iteration 30, Jacobian Loss: 0.00928619597107172
Iteration 35, Jacobian Loss: 0.00878030527383089
Iteration 40, Jacobian Loss: 0.008310753852128983
Iteration 45, Jacobian Loss: 0.0078733516857028
Iteration 50, Jacobian Loss: 0.007470395416021347
Iteration 55, Jacobian Loss: 0.007089228834956884
Iteration 60, Jacobian Loss: 0.00671670725569129
Iteration 65, Jacobian Loss: 0.00635827612131834
Iteration 70, Jacobian Loss: 0.006011292804032564
Iteration 75, Jacobian Loss: 0.005681861191987991
Iteration 80, Jacobian Loss: 0.00535724638029933
Iteration 85, Jacobian Loss: 0.0050449129194021225
Iteration 90, Jacobian Loss: 0.004739170428365469
Iteration 95, Jacobian Loss: 0.004438924137502909


### Add collision loss component

In [3]:
def compute_sdf(mesh, grid_resolution=100):
    """
    Compute the signed-distance function for a given mesh.

    Parameters:
    - mesh_path (str): Path to the mesh file (.obj, .ply, etc.)
    - grid_resolution (int): The resolution of the grid in each dimension.

    Returns:
    - np.ndarray: The SDF grid values shaped in a 3D array.
    - np.ndarray: The grid points as a 3D meshgrid of coordinates.
    """
    # Define the bounds of the grid based on the mesh bounds
    bounds = mesh.bounds
    grid_min, grid_max = bounds[0], bounds[1]

    # Create a grid
    x = np.linspace(grid_min[0], grid_max[0], grid_resolution)
    y = np.linspace(grid_min[1], grid_max[1], grid_resolution)
    z = np.linspace(grid_min[2], grid_max[2], grid_resolution)
    xv, yv, zv = np.meshgrid(x, y, z, indexing='ij')
    grid_points = np.column_stack((xv.ravel(), yv.ravel(), zv.ravel()))

    # Compute the signed distances
    sdf = trimesh.proximity.signed_distance(mesh, grid_points)

    # Reshape the results to a 3D grid
    sdf_grid = sdf.reshape((grid_resolution, grid_resolution, grid_resolution))

    return sdf_grid, (xv, yv, zv)


def trilinear_interpolation(grid, points, grid_min, grid_max):
    """
    Perform trilinear interpolation on a 3D grid at specified points.
    
    Parameters:
    - grid (torch.Tensor): The 3D tensor representing the SDF grid.
    - points (torch.Tensor): Nx3 tensor of points where SDF values need to be interpolated.
    - grid_min (torch.Tensor): The minimum coordinate of the SDF grid in 3D space.
    - grid_max (torch.Tensor): The maximum coordinate of the SDF grid in 3D space.
    
    Returns:
    - torch.Tensor: Interpolated SDF values at the given points.
    """
    # Normalize points to the grid scale
    grid_size = torch.tensor(grid.shape).float() - 1
    points_scaled = (points - grid_min) / (grid_max - grid_min) * grid_size

    # Compute the indices of the grid points to interpolate
    x0 = points_scaled.floor().clamp(0, grid_size[0])
    x1 = (x0 + 1).clamp(0, grid_size[0])
    y0 = points_scaled.floor().clamp(0, grid_size[1])
    y1 = (y0 + 1).clamp(0, grid_size[1])
    z0 = points_scaled.floor().clamp(0, grid_size[2])
    z1 = (z0 + 1).clamp(0, grid_size[2])

    # Interpolation weights
    xd = (points_scaled[:, 0] - x0)
    yd = (points_scaled[:, 1] - y0)
    zd = (points_scaled[:, 2] - z0)

    # Fetch grid values at corner points
    def get_val(x, y, z):
        return grid[x.long(), y.long(), z.long()]

    # Trilinear interpolation
    c00 = get_val(x0, y0, z0) * (1 - xd) + get_val(x1, y0, z0) * xd
    c01 = get_val(x0, y0, z1) * (1 - xd) + get_val(x1, y0, z1) * xd
    c10 = get_val(x0, y1, z0) * (1 - xd) + get_val(x1, y1, z0) * xd
    c11 = get_val(x0, y1, z1) * (1 - xd) + get_val(x1, y1, z1) * xd

    c0 = c00 * (1 - yd) + c10 * yd
    c1 = c01 * (1 - yd) + c11 * yd

    c = c0 * (1 - zd) + c1 * zd
    return c

In [4]:
ALPHA = 1.0
BETA = 0.1
N_ITER = 100

if __name__ == '__main__':
    # Load meshes, setup tensors, etc.
    mesh = trimesh.load('results/optim_in/orig_front_shirt_0.001.ply')
    vertices = torch.tensor(mesh.vertices, dtype=torch.float32, requires_grad=True)
    triangles = torch.tensor(mesh.faces, dtype=torch.int32)

    parameterized_mesh = trimesh.load('results/optim_in/param_front_shirt.obj')
    parameterized_vertices = torch.tensor(parameterized_mesh.vertices, dtype=torch.float32)

    # Compute Jacobians for all triangles in both meshes
    initial_jacobians = compute_jacobians(vertices, triangles, warp_direction, weft_direction)
    target_jacobians = compute_jacobians(parameterized_vertices, triangles, warp_direction, weft_direction)

    # Set parameters
    grid_resolution = 100
    warp_direction = torch.tensor([1, 0, 0], dtype=torch.float32)
    weft_direction = torch.tensor([0, 1, 0], dtype=torch.float32)

    # Compute the SDF of the body mesh before the loop
    sdf_grid, grid_points = compute_sdf(mesh, grid_resolution=10)
    sdf_grid_tensor = torch.tensor(sdf_grid, dtype=torch.float32)#.to(device)
    sdf_min = torch.tensor(grid_points[0].min(axis=0), dtype=torch.float32)#.to(device)
    sdf_max = torch.tensor(grid_points[0].max(axis=0), dtype=torch.float32)#.to(device)

    print('Calculated SDF and Jacobians, starting the optimization...')

    optimizer = torch.optim.Adam([vertices], lr=0.01)

    for iteration in range(N_ITER):
        optimizer.zero_grad()
        
        # Compute Jacobians and SDF based loss
        current_jacobians = compute_jacobians(vertices, triangles, warp_direction, weft_direction)
        jacobian_loss = torch.norm(current_jacobians)  # Simplified
        sdf_values = trilinear_interpolation(sdf_grid_tensor, vertices, sdf_min, sdf_max)
        collision_loss = torch.mean(torch.relu(-sdf_values))  # Assuming SDF is negative inside the mesh

        total_loss = ALPHA * jacobian_loss + BETA * collision_loss
        
        total_loss.backward()
        optimizer.step()

        if iteration % 5 == 0:
            print(f"Iteration {iteration}, Total Loss: {total_loss.item()}")

        if total_loss.item() < 0.01:
            print("Convergence reached")
            break

final_vertices = vertices.detach().cpu().numpy()
optimized_mesh = trimesh.Trimesh(vertices=final_vertices, faces=triangles.cpu().numpy())
optimized_mesh.export('optimized_mesh.ply')

Calculated SDF and Jacobians, starting the optimization...


RuntimeError: The size of tensor a (3) must match the size of tensor b (10) at non-singleton dimension 1

## Solving a system of equations

In [8]:
import numpy as np
from scipy.optimize import minimize
import trimesh

# Function to calculate stretch ratios in warp and weft directions
def calculate_jacobians(vertices, triangles, warp_direction, weft_direction):
    jacobians_warp = []
    jacobians_weft = []
    
    for inds in triangles:
        v0, v1, v2 = vertices[inds]
        edge1 = v1 - v0
        edge2 = v2 - v0

        edge1_warp = np.dot(edge1, warp_direction)
        edge2_warp = np.dot(edge2, warp_direction)
        edge1_weft = np.dot(edge1, weft_direction)
        edge2_weft = np.dot(edge2, weft_direction)

        stretch_warp = np.sqrt(edge1_warp**2 + edge2_warp**2)
        stretch_weft = np.sqrt(edge1_weft**2 + edge2_weft**2)

        jacobians_warp.append(stretch_warp)
        jacobians_weft.append(stretch_weft)

    return np.array(jacobians_warp), np.array(jacobians_weft)

# Define the Jacobian constraints for the optimization
def jacobian_constraint(vertices):
    vertices = vertices.reshape(-1, 3)
    j_constraints = []
    
    current_jacobians_warp, current_jacobians_weft = calculate_jacobians(vertices, triangles, warp_direction, weft_direction)

    for current_warp, target_warp, initial_warp, current_weft, target_weft, initial_weft in zip(current_jacobians_warp, target_jacobians_warp, initial_jacobians_warp, current_jacobians_weft, target_jacobians_weft, initial_jacobians_weft):
        # Normalize by initial jacobians and target ratios
        j_constraints.append((current_warp / initial_warp) - (target_warp / initial_warp))
        j_constraints.append((current_weft / initial_weft) - (target_weft / initial_weft))

    return np.array(j_constraints)


if __name__ == '__main__':
    # Load your garment meshes
    body_mesh = trimesh.load('results/tl_out/orig_body.ply')
    garment_mesh = trimesh.load('results/optim_in/orig_front_shirt_0.001.ply')
    parameterized_mesh = trimesh.load('results/optim_in/param_front_shirt_0.001.obj')

    garment_vertices = garment_mesh.vertices
    param_vertices = parameterized_mesh.vertices
    triangles = garment_mesh.faces

    # Calculate initial and target Jacobians
    initial_jacobians_warp, initial_jacobians_weft = calculate_jacobians(garment_vertices, triangles, warp_direction, weft_direction)
    target_jacobians_warp, target_jacobians_weft = calculate_jacobians(param_vertices, triangles, warp_direction, weft_direction)

    warp_direction = np.array([1, 0, 0])
    weft_direction = np.array([0, 1, 0])
    # Constraints dictionary
    constraints = {'type': 'eq', 'fun': jacobian_constraint}

    # Objective function to minimize the deformation
    def objective_function(vertices, initial_vertices):
        return np.sum((vertices - initial_vertices.flatten())**2)

    # Running the optimization
    result = minimize(objective_function, garment_vertices.flatten(), args=(garment_vertices.flatten(),), constraints=[constraints], method='SLSQP', options={'disp': True})

    # Check the results
    if result.success:
        updated_vertices = result.x.reshape(-1, 3)
        garment_vertices.vertices = updated_vertices
        print("Optimization succeeded.")
    else:
        print("Optimization failed:", result.message)

More equality constraints than independent variables    (Exit mode 2)
            Current function value: 0.0
            Iterations: 1
            Function evaluations: 2140
            Gradient evaluations: 1
Optimization failed: More equality constraints than independent variables
